# IE7615 – Milestone 4 Demo
Loads our trained YOLOv8s weights (`best.pt`) and runs detection. No training happens here.

Each detected object is shown with its Object ID and item name (e.g. `OBJ033 Duck`) and a box around the object itself.

- **Single-object image:** cell (a)
- **Multi-object image:** cell (b)

Run the cells in order. Cells (a) and (b) show an upload button; pick an image and the result prints below.

In [ ]:
!pip install -q ultralytics

Load the weights from our GitHub repo (works for anyone), or from Drive if you have the results folder.

In [ ]:
from ultralytics import YOLO
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

import os

USE_GITHUB = True
GITHUB_WEIGHTS = "https://github.com/adityakusetti/IE7615-Discriminative-Project-Group1/raw/main/best.pt"

if USE_GITHUB:
    os.system(f"wget -q -O best.pt {GITHUB_WEIGHTS}")
    WEIGHTS = "best.pt"
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WEIGHTS = "/content/drive/MyDrive/IE7615_M3_results/best.pt"

model = YOLO(WEIGHTS)
print(len(model.names), "classes:", model.names[0], "...", model.names[len(model.names) - 1])

In [ ]:
TILE = 224
CORNER = True   # single photos go in the corner of a 2x2-sized gray canvas (set from our single-object check)

def single_input(img):
    if not CORNER:
        return img
    c = Image.new("RGB", (448, 448), (114, 114, 114)); c.paste(img, (0, 0)); return c


def detect(path, single=False, conf=0.4):
    orig = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(orig)
    lw = max(2, orig.size[0] // 150)
    if single:
        # center-crop to a square, resize to 224 like the dataset photos, predict, map the box back
        w, h = orig.size; s = min(w, h); ox, oy = (w - s) // 2, (h - s) // 2
        sq = orig.crop((ox, oy, ox + s, oy + s)).resize((TILE, TILE))
        r = model.predict(single_input(sq), imgsz=672, conf=0.05, agnostic_nms=True, verbose=False)[0]
        if not len(r.boxes):
            print("no object detected"); return
        k = int(r.boxes.conf.argmax())
        if float(r.boxes.conf[k]) < 0.5:
            print(f"not one of the 73 known objects (best guess: {model.names[int(r.boxes.cls[k])]} at {float(r.boxes.conf[k]):.2f})")
            return
        f = s / TILE
        x1, y1, x2, y2 = [v * f for v in r.boxes.xyxy[k].tolist()]
        found = [(model.names[int(r.boxes.cls[k])], float(r.boxes.conf[k]), (x1 + ox, y1 + oy, x2 + ox, y2 + oy))]
    else:
        r = model.predict(orig, imgsz=672, conf=conf, agnostic_nms=True, verbose=False)[0]
        found = [(model.names[int(b.cls)], float(b.conf), tuple(b.xyxy[0].tolist())) for b in r.boxes]
        found.sort(key=lambda t: (round(t[2][1] / 100), t[2][0]))
        print(f"{len(found)} objects found")
    for name, c, (x1, y1, x2, y2) in found:
        d.rectangle([x1, y1, x2, y2], outline="red", width=lw)
        d.text((x1 + 4, y1 + 4), f"{name} {c:.2f}", fill="yellow")
        print(f"{name:<24} {c:.2f}  ({int(x1)}, {int(y1)}, {int(x2)}, {int(y2)})")
    plt.figure(figsize=(8, 8)); plt.imshow(orig); plt.axis("off"); plt.show()
    return found

### a) Single object

In [ ]:
# click "Choose Files" and pick a single-object photo
from google.colab import files
for name in files.upload():
    detect(name, single=True)

### b) Multiple objects

In [ ]:
# click "Choose Files" and pick an image with several objects (e.g. a 2x2 or 3x3 grid)
from google.colab import files
for name in files.upload():
    detect(name)
